In [1]:
import os
import numpy as np
import torch
import cv2
from torch import nn
from pathlib import Path
from collections import defaultdict
from ultralytics import YOLO

In [ ]:
! nvidia-smi

In [2]:
import ultralytics
ultralytics.checks()

Ultralytics 8.3.168  Python-3.10.18 torch-2.7.1+cu126 CUDA:0 (NVIDIA GeForce RTX 3060, 12287MiB)
Setup complete  (16 CPUs, 31.9 GB RAM, 223.9/476.4 GB disk)


In [3]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
device

device(type='cuda')

configure

In [6]:
PATH_WEIGHT = r'weights_sdu_v3\best_sdu_v3.pt'  #r'yolo11m.pt'
PATH_VIDEO_TEST = r'E:\TRACKIING\video\IMG_2529_.mp4' #DJIG0103.mp4'

Export модели yolo11n-cls для reid, нужны доп библиотеки

In [ ]:
! pip uninstall tensorrt nvidia-tensorrt
! pip cache purge
! pip install tensorrt==10.1.*

^C


In [ ]:
! pip install tensorrt

In [5]:
import tensorrt
tensorrt.__version__

'10.13.3.9'

In [5]:
model_cls = YOLO('yolo11n-cls.pt')

In [6]:
head = model_cls.model.model[-1]
pool = nn.Sequential(nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(start_dim=1))
pool.f, pool.i = head.f, head.i
model_cls.model.model[-1] = pool

model_cls.export(format='engine', half=True, dynamic=True, batch=32)

WARNING TensorRT requires GPU export, automatically assigning device=0
Ultralytics 8.3.168  Python-3.10.18 torch-2.7.1+cu126 CUDA:0 (NVIDIA GeForce RTX 3060, 12287MiB)
YOLO11n-cls summary (fused): 45 layers, 1,197,064 parameters, 0 gradients, 2.9 GFLOPs

PyTorch: starting from 'yolo11n-cls.pt' with input shape (32, 3, 224, 224) BCHW and output shape(s) (32, 256) (5.5 MB)

ONNX: starting export with onnx 1.17.0 opset 19...
ONNX: slimming with onnxslim 0.1.65...
ONNX: export success  2.2s, saved as 'yolo11n-cls.onnx' (4.6 MB)

TensorRT: starting export with TensorRT 10.13.3.9...
TensorRT: input "images" with shape(-1, 3, -1, -1) DataType.FLOAT
TensorRT: output "output0" with shape(-1, 256) DataType.FLOAT
TensorRT: building FP16 engine as yolo11n-cls.engine
TensorRT: export success  363.6s, saved as 'yolo11n-cls.engine' (4.8 MB)

Export complete (364.8s)
Results saved to C:\TASK_DETECT_DRONE\CODES_DETECT_DRONE\YOLO+calc_distance
Predict:         yolo predict task=classify model=yolo11n-cl

'yolo11n-cls.engine'

define function

In [4]:
def load_model(path_weight):
    if os.path.isfile(path_weight):
        model = YOLO(path_weight)
        return model
    else:
        print('Weight model not load')

In [10]:
current_tracks = []
selected_id = None

def mouse_click(event, x, y, flags, param):
    global selected_id, frame

    if event == cv2.EVENT_LBUTTONDOWN:
        # print(f'Click to coordinate: ({x}, {y})')

        # Проверяем попал ли клик в какойто bb
        for bbox in current_tracks:
            
            x1, y1, x2, y2, track_id = map(int, bbox[:5])
            if x1 <= x < x2 and y1 <= y <= y2:
                print(f'Choice object: {bbox}')
                selected_id = int(track_id)
                print(f'Choice object with ID: {selected_id}')
                break

Process

with press key in object

In [9]:
current_tracks = []
selected_id = None

def mouse_click(event, x, y, flags, param):
    global selected_id, frame

    if event == cv2.EVENT_LBUTTONDOWN:
        # print(f'Click to coordinate: ({x}, {y})')

        # Проверяем попал ли клик в какойто bb
        for bbox in current_tracks:
            
            x1, y1, x2, y2, track_id = map(int, bbox[:5])
            if x1 <= x < x2 and y1 <= y <= y2:
                print(f'Choice object: {bbox}')
                selected_id = int(track_id)
                print(f'Choice object with ID: {selected_id}')
                break

cap = cv2.VideoCapture(0)  #PATH_VIDEO_TEST

model = load_model(PATH_WEIGHT)
cv2.namedWindow('YOLO11 Tracking')
cv2.setMouseCallback("YOLO11 Tracking", mouse_click)


while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        print('Not read frame!')
        break
    results = model.track(frame, 
                          conf = 0.4,
                          iou = 0.2,
                          imgsz = 1024,
                          persist=True, 
                          verbose = False,
                         
                          tracker=r'conf_trackers\botsort_custom.yaml')[0]
    if results.boxes and results.boxes.is_track:
        boxesxy = results.boxes.xyxy.cpu()
        # boxes = results.boxes.xywh.cpu()
        track_ids = results.boxes.id.int().cpu().tolist()
        confideces = results.boxes.conf.cpu().tolist()
        clas = results.boxes.cls.cpu().tolist()
        
        #plot the tracks
        for box, track_id, conf, cl_ in zip(boxesxy, track_ids, confideces, clas):
            x, y, x2, y2 = map(int, box)
            current_tracks.append((x, y, x2, y2, track_id))
            color = (0, 0, 255)
            if selected_id is not None and selected_id == track_id:
                color_ = (255, 0, 0)
                cv2.rectangle(frame, (x, y), (x2, y2), color_, 2)
                cv2.putText(frame, f'Selected_{track_id}', (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.75, color_, 2)
            else:
                cv2.rectangle(frame, (x, y), (x2, y2), color, 1)
                cv2.putText(frame, f'{track_id}', (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
        

    cv2.imshow('YOLO11 Tracking', frame)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('r'):
        selected_id = None
        print('Drop select object...')

    if cv2.waitKey(0) & 0xFF == ord('q'):
        print('Press q')
        break
        

cap.release()
cv2.destroyAllWindows()


Press q


In [ ]:
track_history = defaultdict(lambda: [])
model = load_model(PATH_WEIGHT)

assert os.path.isfile(PATH_VIDEO_TEST), 'Not file vide'

cap = cv2.VideoCapture(PATH_VIDEO_TEST)  #PATH_VIDEO_TEST

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        print('Not read frame!')
        break
    results = model.track(frame, 
                          conf = 0.4,
                          iou = 0.2,
                          imgsz = 1024,
                          persist=True, 
                          verbose = False,
                         
                          tracker='botsort.yaml')[0]
    if results.boxes and results.boxes.is_track:
        boxes = results.boxes.xywh.cpu()
        track_ids = results.boxes.id.int().cpu().tolist()
 

        #visual the result on the frame
        frame = results.plot()

        #plot the tracks
        for box, track_id in zip(boxes, track_ids):
            x, y, w, h = box
            track = track_history[track_id]
            track.append((float(x), float(y))) # y, x center point
            if len(track) > 30: # retain 30 tracks for 30 frames
                track.pop(0)

            points = np.hstack(track).astype(np.int32).reshape((-1, 1, 2))
            cv2.polylines(frame, [points], isClosed=False, color=(230, 230, 230), thickness=1)

        cv2.imshow('YOLO11 Tracking', frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            print('Press q')
            break
    else:
        print('Not boxes or is_track')
        continue

cap.release()
cv2.destroyAllWindows()


Press q


In [7]:
model = load_model(PATH_WEIGHT)

# assert os.path.isfile(PATH_VIDEO_TEST), 'Not file vide'

cap = cv2.VideoCapture(0)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        print('Not read frame!')
        break
    results = model.track(frame, 
                    conf = 0.4,
                    iou = 0.2,
                    imgsz = 1024,
                    verbose = False,
                    device=device,
                    tracker=r'conf_trackers\botsort_custom.yaml'
                    )[0]
    annotaited_frame = results.plot()
    
    cv2.imshow('YOLO11 Tracking', annotaited_frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        print('Press q')
        break
    

cap.release()
cv2.destroyAllWindows()


Press q


Test from boxmot tracker Botsort

In [5]:
from boxmot import BotSort

In [6]:
PATH_WEIGHT = r'weights_sdu_v3\best_sdu_v3.pt'
model = load_model(PATH_WEIGHT) # load YOLO model SDU my

In [24]:
PATH_WEIGHT_BOTSORT = r'weights_botsort_boxmot\osnet_x0_25_msmt17.pt'
tracker = BotSort(reid_weights=Path(PATH_WEIGHT_BOTSORT), device='cuda:0', half=False)

2025-09-23 10:01:58.669 | INFO     | boxmot.utils.torch_utils:select_device:78 - Yolo Tracking v15.0.1 🚀 Python-3.10.18 torch-2.7.1+cu126
CUDA:0 (NVIDIA GeForce RTX 3060, 12287MiB)
2025-09-23 10:01:58.671 | ERROR    | boxmot.appearance.backends.base_backend:download_model:152 - Found existing ReID weights at weights_botsort_boxmot\osnet_x0_25_msmt17.pt; skipping download.
2025-09-23 10:01:58.930 | SUCCESS  | boxmot.appearance.reid.registry:load_pretrained_weights:64 - Loaded pretrained weights from weights_botsort_boxmot\osnet_x0_25_msmt17.pt


In [22]:
def detect(frame):
    detection = []
    threshold_conf = 0.25
    result = model(frame,
                   imgsz=1024,
                   conf=threshold_conf,
                   iou=0.4,
                   device='cpu',
                   verbose=False)
    boxes = result[0].boxes.xyxy.cpu()
    confs = result[0].boxes.conf.cpu()
    idx_cls = [int(i) for i in list(result[0].boxes.cls.cpu())]
    for box, conf, id_cl in zip(boxes, confs, idx_cls):
        x1 = box[0]
        y1 = box[1]
        x2 = box[2]
        y2 = box[3]
        detection.append((x1, y1, x2, y2, conf, id_cl))
    return np.array(detection, dtype=np.float32)


process

In [25]:
PATH_VIDEO_TEST =  0 #r'E:\TRACKIING\video\IMG_2529_.mp4'  #r'E:\TRACKIING\video\DJIG0103.mp4'

track_history = defaultdict(lambda: [])
color = {
    '0': (255, 0, 0),
    '1': (0, 255, 0),
    '2': (230, 230, 230),
    '3': (255, 0, 0),
}
cap = cv2.VideoCapture(PATH_VIDEO_TEST)

frame_id = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        print('Not read frame!')
        break

    det = detect(frame)

    tracks = tracker.update(det, frame)

    for track in tracks:
        x1, y1, x2, y2, track_id, conf, id_c, _ = map(int, track[:8])
        color_bb = color.get(str(id_c), (0, 0, 0))
        cv2.rectangle(frame, (x1, y1), (x2, y2), color_bb, 1)
        
        cv2.putText(frame, f'{track_id}', (x1, y1 -10), cv2.FONT_HERSHEY_COMPLEX, 0.75, color_bb, 2)

        #save history move
        center = ((x1 + x2)//2, (y1 + y2)//2)
        track_history[track_id].append(center)

        # ploting trajectory
        # for i in range(1, len(track_history[track_id])):
        #     cv2.line(frame, track_history[track_id][i - 1], track_history[track_id][i], (0, 0, 255), 2)

    
    
    cv2.imshow('YOLO11 Tracking', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        print('Press q')
        break
    

cap.release()
cv2.destroyAllWindows()


2025-09-23 10:02:24.196 | WARNING  | boxmot.motion.cmc.ecc:apply:99 - Affine matrix could not be generated: OpenCV(4.12.0) D:\a\opencv-python\opencv-python\opencv\modules\video\src\ecc.cpp:574: error: (-7:Iterations do not converge) NaN encountered. in function 'cv::findTransformECC'
. Returning identity
2025-09-23 10:02:24.323 | WARNING  | boxmot.motion.cmc.ecc:apply:99 - Affine matrix could not be generated: OpenCV(4.12.0) D:\a\opencv-python\opencv-python\opencv\modules\video\src\ecc.cpp:589: error: (-7:Iterations do not converge) The algorithm stopped before its convergence. The correlation is going to be minimized. Images may be uncorrelated or non-overlapped in function 'cv::findTransformECC'
. Returning identity
2025-09-23 10:02:24.458 | WARNING  | boxmot.motion.cmc.ecc:apply:99 - Affine matrix could not be generated: OpenCV(4.12.0) D:\a\opencv-python\opencv-python\opencv\modules\video\src\ecc.cpp:589: error: (-7:Iterations do not converge) The algorithm stopped before its conver

Press q
